# 04 — Error Analysis

Deep-dive analyses:
1. Accuracy by sentence-length bucket
2. Per-dependency-relation accuracy
3. Long-distance arc stress-test (distance > 5)
4. Top confusion pairs

In [1]:
# === Kaggle / Colab setup (skip if running locally) ===
import os, subprocess, sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB  = "google.colab" in sys.modules

if IS_KAGGLE or IS_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "../requirements.txt"], check=True)
    # Kaggle ships pandas built against numpy 2.x, but spacy 3.7.5 pins numpy<2.
    # Force-reinstall pandas+numpy together so binary ABI matches.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--force-reinstall", "--no-deps",
                    "numpy<2", "pandas>=2.0,<2.3"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "en_core_web_trf"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "ru_core_news_lg"], check=True)
    subprocess.run([sys.executable, "../scripts/download_data.py"],
                   check=True)
    print("Setup done.")
else:
    print("Local env detected — skipping cloud setup.")


Local env detected — skipping cloud setup.


In [2]:
import sys
sys.path.insert(0, "..")

from collections import Counter
from pathlib import Path
import pandas as pd
from tqdm import tqdm

from src.data import load_sentences, bucket_by_length
from src.parsers import SpacyParser, StanzaParser
from src.metrics import uas, las, Gold, Prediction, _is_punct

SENTS = {
    "en": load_sentences(Path("../data/en_ewt_test.conllu")),
    "ru": load_sentences(Path("../data/ru_syntagrus_test.conllu")),
}
PARSERS = {
    "en": [SpacyParser("en_core_web_trf"), StanzaParser("en")],
    "ru": [SpacyParser("ru_core_news_lg"), StanzaParser("ru")],
}
print("Parsers loaded.")

Parsers loaded.


In [3]:
# Re-parse everything and keep predictions
CHUNK = 100
PARSES = {}  # (lang, parser_name) -> list[Prediction]

for lang, sents in SENTS.items():
    for parser in PARSERS[lang]:
        toks = [s.tokens for s in sents]
        results = []
        for i in tqdm(range(0, len(toks), CHUNK), desc=f"{lang}-{parser.name}"):
            results.extend(parser.parse(toks[i:i+CHUNK]))
        PARSES[(lang, parser.name)] = [Prediction(r.heads, r.deprels) for r in results]

print("Done parsing.")

en-spacy:en_core_web_trf:   0%|          | 0/21 [00:00<?, ?it/s]/Users/aleksandrgavkovskij/programming/nlp_case_study/.venv/lib/python3.12/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
ru-stanza:ru: 100%|██████████| 88/88 [03:07<00:00,  2.14s/it]

Done parsing.


In [4]:
# 1. Accuracy by sentence length bucket
rows = []
for lang, sents in SENTS.items():
    buckets = bucket_by_length(sents)
    sent_to_idx = {id(s): i for i, s in enumerate(sents)}
    for bucket_name, bucket_sents in buckets.items():
        idxs = [sent_to_idx[id(s)] for s in bucket_sents]
        if not idxs:
            continue
        for (lang_, pname), preds in PARSES.items():
            if lang_ != lang:
                continue
            sub_pred = [preds[i] for i in idxs]
            sub_gold = [Gold(sents[i].heads, sents[i].deprels) for i in idxs]
            rows.append({
                "lang": lang, "parser": pname, "bucket": bucket_name,
                "n": len(idxs),
                "uas": round(uas(sub_pred, sub_gold), 4),
                "las": round(las(sub_pred, sub_gold), 4),
            })

df_len = pd.DataFrame(rows)
df_len.to_csv("../results/accuracy_by_length.csv", index=False)
print("Saved accuracy_by_length.csv")
df_len

Saved accuracy_by_length.csv


,lang,parser,bucket,n,uas,las
0,en,spacy:en_core_web_trf,1-10,1164,0.6430,0.5073
1,en,stanza:en,1-10,1164,0.9090,0.8826
2,en,spacy:en_core_web_trf,11-20,551,0.6173,0.4347
3,en,stanza:en,11-20,551,0.9205,0.8955
4,en,spacy:en_core_web_trf,21-40,308,0.6022,0.4355
5,en,stanza:en,21-40,308,0.8997,0.8771
6,en,spacy:en_core_web_trf,40+,54,0.5928,0.4249
7,en,stanza:en,40+,54,0.8885,0.8662
8,ru,spacy:ru_core_news_lg,1-10,2485,0.9035,0.8509
9,ru,stanza:ru,1-10,2485,0.9409,0.9057


In [5]:
# 2. Per-relation accuracy
rows = []
for (lang, pname), preds in PARSES.items():
    sents = SENTS[lang]
    per_rel_total: Counter = Counter()
    per_rel_correct: Counter = Counter()
    for pred, sent in zip(preds, sents):
        for ph, pd_, gh, gd in zip(pred.heads, pred.deprels, sent.heads, sent.deprels):
            if _is_punct(gd):
                continue
            key = gd.split(":")[0]
            per_rel_total[key] += 1
            if ph == gh and pd_.split(":")[0] == key:
                per_rel_correct[key] += 1
    for rel, total in per_rel_total.items():
        rows.append({
            "lang": lang, "parser": pname, "relation": rel,
            "n": total,
            "accuracy": round(per_rel_correct[rel] / total, 4) if total else 0.0,
        })

df_rel = pd.DataFrame(rows).sort_values(["lang", "relation", "parser"])
df_rel.to_csv("../results/accuracy_by_relation.csv", index=False)
print("Saved accuracy_by_relation.csv")
df_rel.head(20)

Saved accuracy_by_relation.csv


,lang,parser,relation,n,accuracy
15,en,spacy:en_core_web_trf,acl,377,0.2361
48,en,stanza:en,acl,377,0.7560
3,en,spacy:en_core_web_trf,advcl,366,0.4645
36,en,stanza:en,advcl,366,0.7732
9,en,spacy:en_core_web_trf,advmod,1314,0.6644
42,en,stanza:en,advmod,1314,0.8691
12,en,spacy:en_core_web_trf,amod,1253,0.8276
45,en,stanza:en,amod,1253,0.8994
25,en,spacy:en_core_web_trf,appos,185,0.5568
58,en,stanza:en,appos,185,0.5081


In [6]:
# 3. Long-distance arc stress-test (distance > 5)
rows = []
for (lang, pname), preds in PARSES.items():
    sents = SENTS[lang]
    lt = lc_uas = lc_las = 0
    for pred, sent in zip(preds, sents):
        for i, (ph, pd_, gh, gd) in enumerate(
            zip(pred.heads, pred.deprels, sent.heads, sent.deprels)
        ):
            if _is_punct(gd):
                continue
            dep_pos = i + 1
            dist = abs(dep_pos - gh) if gh != 0 else dep_pos
            if dist <= 5:
                continue
            lt += 1
            if ph == gh:
                lc_uas += 1
            if ph == gh and pd_.split(":")[0] == gd.split(":")[0]:
                lc_las += 1
    rows.append({
        "lang": lang, "parser": pname,
        "n_long_arcs": lt,
        "uas_long": round(lc_uas / lt, 4) if lt else 0.0,
        "las_long": round(lc_las / lt, 4) if lt else 0.0,
    })

df_long = pd.DataFrame(rows)
df_long.to_csv("../results/long_distance_stress.csv", index=False)
print("Saved long_distance_stress.csv")
df_long

Saved long_distance_stress.csv


,lang,parser,n_long_arcs,uas_long,las_long
0,en,spacy:en_core_web_trf,2098,0.3880,0.3051
1,en,stanza:en,2098,0.7598,0.7331
2,ru,spacy:ru_core_news_lg,15727,0.7415,0.7021
3,ru,stanza:ru,15727,0.8398,0.8135


In [7]:
# 4. Top confusion pairs (gold label → predicted label)
rows = []
for (lang, pname), preds in PARSES.items():
    sents = SENTS[lang]
    conf: Counter = Counter()
    for pred, sent in zip(preds, sents):
        for pd_, gd in zip(pred.deprels, sent.deprels):
            if _is_punct(gd):
                continue
            p_label = pd_.split(":")[0]
            g_label = gd.split(":")[0]
            if p_label != g_label:
                conf[(g_label, p_label)] += 1
    for (g, p), n in conf.most_common(20):
        rows.append({"lang": lang, "parser": pname, "gold": g, "predicted": p, "count": n})

df_conf = pd.DataFrame(rows)
df_conf.to_csv("../results/confusion_top.csv", index=False)
print("Saved confusion_top.csv")
df_conf.head(20)

Saved confusion_top.csv


,lang,parser,gold,predicted,count
0,en,spacy:en_core_web_trf,case,prep,1783
1,en,spacy:en_core_web_trf,obj,dobj,1050
2,en,spacy:en_core_web_trf,obl,pobj,963
3,en,spacy:en_core_web_trf,nmod,pobj,715
4,en,spacy:en_core_web_trf,nmod,poss,372
5,en,spacy:en_core_web_trf,mark,aux,368
6,en,spacy:en_core_web_trf,cop,root,312
7,en,spacy:en_core_web_trf,acl,relcl,207
8,en,spacy:en_core_web_trf,advmod,neg,199
9,en,spacy:en_core_web_trf,root,acomp,154


## What we learned (from this run)

Four error-analysis CSVs written to `results/`:

**1. `accuracy_by_length.csv` — degradation with sentence length (UAS):**
- RU: spaCy 0.904 (1–10) → 0.865 (40+); Stanza 0.941 → 0.913. Gap widens from 3.7 pt to 4.9 pt.
- EN: spaCy 0.643 → 0.593; Stanza 0.909 → 0.889. Both degrade by ~5 pt across buckets, but spaCy starts so much lower that the absolute gap (~30 pt) barely moves.

**2. `accuracy_by_relation.csv` — per-relation accuracy:**
- **EN:** spaCy scores near-zero on `case`, `cop`, `obj`, `obl`, `nmod`, `flat`, `fixed`, `iobj`, `list` — artefact of CLEAR-style labels being counted as wrong. Confirms the EN LAS caveat.
- **RU (fair UD-vs-UD comparison):** Stanza's biggest wins: `compound` (0.12 → 0.77, +65 pt), `expl` (0.06 → 0.73, +67 pt), `vocative` (0.00 → 0.59, +59 pt), `parataxis` (0.47 → 0.70, +24 pt), `flat` (0.72 → 0.91, +19 pt), `obl` (0.71 → 0.87, +16 pt). **spaCy wins on `xcomp` (0.936 vs 0.867, +6.9 pt) and `iobj` (0.77 vs 0.73, +4.0 pt)** — short-range lexically-cued relations where local features suffice.

**3. `long_distance_stress.csv` — arcs with |head − dep| > 5:**
- EN: spaCy 0.388 UAS vs Stanza 0.760 — **+37 pt** gap (vs +29 pt overall).
- RU: spaCy 0.742 vs Stanza 0.840 — **+9.8 pt** gap (vs +4.1 pt overall). The long-arc gap is more than 2× the aggregate gap.

**4. `confusion_top.csv` — top 20 gold→predicted label mismatches per parser:**
- spaCy EN: dominated by CLEAR→UD label drift (`case→prep` 1783, `obj→dobj` 1050, `obl→pobj` 963). Not "real" errors — they're a scoring mismatch.
- spaCy RU: real attachment errors — `obl→advmod` 925, `obl→nmod` 797, `obl→obj` 608. Misrouting of oblique arguments is spaCy RU's signature failure mode.
- Stanza RU: much smaller counts and more symmetric (`obl↔nmod` ~540 each way, `nmod↔det` drift) — boundary cases between closely-related UD relations, not structural misroutings.

**Takeaway:** the accuracy gap is not uniform — it concentrates in long arcs, long sentences, and globally-constrained relations. Notebook 06 adds the projectivity slice, which tightens this picture further.
